 ## 🧮 Lab 2: Algebra Liniowa dla Informatyki Kwantowej
**Katedra: Informatyki, Mechaniki i Robotyki | wersja 0.1**  

## 🔑 Karty Referencyjne (Mapping Dirac ↔ NumPy)
| Działanie matematyczne | Notacja Diraca | Wtyczka NumPy | Uwaga kluczowa |
|------------------------|----------------|---------------|----------------|
| Iloczyn wewnętrzny     | $\braket{u|v}$ | `np.vdot(u, v)` | `vdot` **automatycznie sprzęga pierwszy argument** |
| Iloczyn zewnętrzny     | $\ket{u}\bra{v}$ | `u[:, None] @ np.conj(v)[None, :]` | Jawne wymiary `[None]` zapobiegają błędom broadcastingu |
| Norm a normalizacja    | $\|\ket{\psi}\|=1$ | `np.linalg.norm(v)` / `v/np.linalg.norm(v)` | Norma $\implies$ warunek Born-a (suma $|c_i|^2=1$) |
| Niezależność liniowa   | kolumny $A$ niezależne | `try: np.linalg.solve(A, np.ones(n))` | Testujemy odwracalność przez niezerowy RHS. Stabilniej niż `det()`. |
| Iloczyn tensorowy      | $\ket{u}\otimes\ket{v}$ | `np.kron(u, v)` | Rozszerza przestrzeń Hilberta: $n \times m$ wymiarów |

## 💡 Wskazówki do rozwiązywania zadań
Poniżej znajdują się wskazówki techniczne, które pomogą Ci w poprawnym wykonaniu obliczeń:

1. **Sprzężenie hermitowskie ($\dagger$):** Pamiętaj o kolejności operacji: najpierw dokonaj transpozycji macierzy, a następnie oblicz sprzężenie zespolone każdego elementu.
2. **Wartości własne:** Aby wyznaczyć $\lambda$, rozwiąż równanie charakterystyczne $\det(M - \lambda I) = 0$.
3. **Wektory własne:** Dla każdej znalezionej wartości $\lambda$, podstaw ją do układu $(M - \lambda I)\ket{v} = \mathbf{0}$. Warto jest znormalizować wektory własne ($\| \ket{v} \| = 1$).
4. **Projektory:** Projektor na stan $\ket{\psi}$ to macierz obliczana jako iloczyn zewnętrzny $\ket{\psi}\bra{\psi}$.

### 🛠️ Mapowanie na funkcje NumPy:
- Sprzężenie hermitowskie: `A.conj().T`
- Wartości i wektory własne: `np.linalg.eigh(A)` (zalecane dla macierzy hermitowskich)
- Iloczyn zewnętrzny (projektor): `np.outer(v, v.conj())` lub `v_col @ v_col.conj().T`
- Mnożenie macierzy: `A @ v`

In [ ]:
import sys
import subprocess

# Lista modułów do sprawdzenia i ewentualnej instalacji
required_modules = ["tabulate", "numpy"]

modules_installed = False

for module in required_modules:
    try:
        __import__(module)
        print(f"Moduł '{module}' jest już zainstalowany.")
    except ImportError:
        print(f"Moduł '{module}' nie został znaleziony. Instalacja...")
        
        # Instalacja modułu w bieżącym środowisku Jupytera
        subprocess.check_call([sys.executable, "-m", "pip", "install", module])
        print(f"Moduł '{module}' został pomyślnie zainstalowany.")
        
        modules_installed = True

# Restart kernela nastąpi tylko wtedy, gdy zainstalowano nowy moduł
if modules_installed:
    print("\nZainstalowano nowe moduły. Restartowanie kernela...")
    try:
        from IPython import get_ipython
        get_ipython().kernel.do_shutdown(True)
    except AttributeError:
        import os
        os._exit(0)
else:
    print("\nWszystkie moduły są gotowe do użycia. Restart nie jest wymagany.")


In [ ]:
import numpy as np
from tabulate import tabulate

# 1. Sprzężenie hermitowskie (macierz sprzężona i przetransponowana)
def dagger(M):
    return M.conj().T
# 2. Operator rzutowy (projektor) dla stanu |v><v|
def projector(v):
    v = v / np.linalg.norm(v)        # upewniamy się, że norma wynosi 1
    v_col = v.reshape(-1, 1)         # przekształcenie w wektor kolumnowy
    return v_col @ dagger(v_col)     # mnożenie |v><v|
# 3. Wartości własne i wektory własne (NumPy zwraca już znormalizowane wektory)
def eigendecomp(M, is_hermitian=False):
    vals, vecs = np.linalg.eigh(M) if is_hermitian else np.linalg.eig(M)
    return vals, vecs
# vals: jednowymiarowa tablica (1D array) zawierająca wartości własne.
# vecs: dwuwymiarowa macierz (2D array), gdzie KOLUMNY są wektorami własnymi.    
# 4. Weryfikacja unitarności (U^dagger @ U = I)
def is_unitary(U, eps=1e-12):
    U = np.asarray(U)
    if U.ndim != 2 or U.shape[0] != U.shape[1]:
        return False
    I = np.eye(U.shape[0], dtype=complex)
    return (np.allclose(U.conj().T @ U, I, rtol=0.0, atol=eps)
        and np.allclose(U @ U.conj().T, I, rtol=0.0, atol=eps))
    
# 4. Pomoc do  wy swietlania macierzy
def wypisz_macierz(tytul, macierz):
    # 1. Mapujemy każdy element na ładny ciąg tekstowy
    tabela_tekstowa = []
    for wiersz in macierz:
        nowy_wiersz = []
        for x in wiersz:
            if np.isclose(x.imag, 0):
                nowy_wiersz.append(f"{x.real:.2f}")  # Chowamy +0j
            else:
                nowy_wiersz.append(f"{x.real:+.2f}{x.imag:+.2f}j")
        tabela_tekstowa.append(nowy_wiersz)
    
    # 2. Drukowanie za pomocą tabulate (używamy formatu 'grid' dla konsoli)
    # Możesz zmienić tablefmt na 'latex_booktabs', aby wygenerować kod do LaTeXa!
    print(f"\n{tytul}:")
    print(tabulate(tabela_tekstowa, tablefmt="grid", stralign="center"))

print("✅ Biblioteki i funkcje pomocnicze zostały załadowane.")

# 💻Zadania domowe

Poniższe zadania mają na celu utrwalenie wiedzy teoretycznej poprzez jej praktyczną weryfikację. Każde zadanie wymaga przeprowadzenia obliczeń analitycznych (ręcznych) oraz ich potwierdzenia w środowisku Python/NumPy.

### 📌 Wskazówki do rozwiązywania zadań

Poniższe wskazówki pomogą Ci przejść od obliczeń ręcznych do ich weryfikacji w języku Python. Pamiętaj, że celem jest zrozumienie mechaniki operacji, a nie tylko uzyskanie wyniku.

### Obliczenia analityczne (ręczne):

Liczby zespolone: Aby pozbyć się jednostki urojonej z mianownika, pomnóż licznik i mianownik przez sprzężenie mianownika. Przy potęgowaniu rozważ zamianę na postać biegunową $re^{i\varphi}$.
Notacja Diraca: Pamiętaj, że $\bra{u}$ to sprzężony wiersz. Przed mnożeniem w $\braket{u}{v}$ każdy element z lewego wektora musi zostać sprzężony.
Iloczyn zewnętrzny: Wynik $\ket{u}\bra{v}$ to macierz. Oblicz go jako mnożenie kolumny przez wiersza: element $(i,j)$ to $u_i \cdot v_j^*$.
Iloczyn tensorowy: Przy $\ket{\psi} \otimes \ket{\phi}$ mnożymy każdy element pierwszego wektora przez cały drugi wektor. Dla wymiarów $2 \times 2$ wynik będzie wektorem o rozmiarze $4$.*

### 📌 Wskazówki implementacyjne

+ Używaj [:, None] lub .reshape(-1, 1), aby wymusić kolumnowe kształty wektorów.
+ Iloczyn wewnętrzny: np.vdot(a, b) (automatycznie sprzęga pierwszy argument).
+ Iloczyn zewnętrzny: a @ np.conj(b).T lub a[:, None] @ b[None, :].
+ Iloczyn tensorowy: np.kron(a, b).

### 📐 Instrukcja dotycząca obliczeń analitycznych (ręcznych)

Wszystkie ręczne wyliczenia, dowody algebraiczne oraz interpretacje wyników **należy umieszczać w komórkach Markdown**, używając formatu LaTeX do zapisu formuł matematycznych. Poniżej znajduje się przewodnik po niezbędnych konstrukcjach:

#### 🔹 Podstawowe konstruktory LaTeX (działające w Jupyter)
| Element | Składnia LaTeX | Efekt renderowany |
|:---|:---|:---|
| **Wektory Diraca** | `\( \ket{0} \)` \| `\( \bra{\psi} \)` | $\ket{0}$ \| $\bra{\psi}$ |
| **Iloczyny wewnętrzne** | `\( \braket{a}{b} \)` \| `\( \braket{a\|b} \)` | $\braket{a|b}$  |
| **Liczby zespolone** | `\( i \)` \| `\( \overline{z} \)` \| `\( |z|^2 \)` | $i$ \| $\overline{z}$ \| $|z|^2$ |
| **Ułamki / pierwiastki** | `\( \frac{a}{b} \)` \| `\( \sqrt{x} \)` | $\frac{a}{b}$ \| $\sqrt{x}$ |
| **Macierze / wektory** | `\[ \begin{pmatrix} 1 \\ i \end{pmatrix} \]` | $$\begin{pmatrix} 1 \\ i \end{pmatrix}$$ |
| **Operacje kwantowe** | `\( \otimes \)` \| `\( \sum_{k=0}^{n} \)` | $\otimes$ \| $\sum_{k=0}^{n}$ |

#### 💡 Wskazówki techniczne:
- Zawsze otaczaj wzory `$` (wierszowy) lub `$$ $$` / `\[ \]` (akapitowy/new line).
- Do wykonywania operacji numerycznych używaj oddzielnych komórek `Code`.
- Wyniki analityczne zapisuj wyraźnie i numeruj, aby łatwo porównać je z outputem Pythona.
- Jeśli `\braket{...}` nie renderuje się poprawnie, alternatywnie użyj: `\( \langle a | b \rangle \)` → $\langle a | b \rangle$.

## Zadanie 1: Operatory i sprzężenie hermitowskie
**Opis:** Wykonaj analizę macierzy operatora $A$ i sprawdź jej podstawowe właściwości.

**Dane:** 
$$A = \begin{bmatrix} 2+i & 1-2i \\ 1+2i & 3 \end{bmatrix}$$

**Analiza analityczna:**
1. Wyznacz macierz sprzężoną transponowaną $A^{\dagger}$.
2. Sprawdź, czy $A$ jest hermitowski ($A = A^{\dagger}$).
3. Oblicz iloczyny $AA^{\dagger}$ oraz $A^{\dagger}A$. Czy są równe?
4. Wyznacz komutator $[A,A^{\dagger}]=AA^{\dagger}-A^{\dagger}A$ i określ, czy $A$ jest **normalny**.
5. Wyznacz wartości własne macierzy $AA^{\dagger}$.

#### ✍️ Rozwiązanie analityczne:

1. **Sprzężenie hermitowskie $A^\dagger$:** ...
2. **Hermitowskość:** ...
3. **Iloczyny:** ...
4. **Komutator:** ...
5. **Wartości własne:** ...


In [ ]:
A = np.array([[2+1j, 1-2j], [1+2j, 3]])

# 1. Wyznacz A_dagger
A_dagger = ...
wypisz_macierz("A_dagger", A_dagger)

# 2. Sprawdź hermitowskość
is_hermitian = ...
print(f"\nCzy A jest hermitowski? {is_hermitian}")

# 3. Iloczyny i 4. Komutator
AA_dag = ...
A_dag_A = ...
commutator = ...

wypisz_macierz("AA_dagger", AA_dag)
wypisz_macierz("A_dag_A", A_dag_A)
print(f"\nCzy A jest normalny? {np.allclose(commutator, 0)}")

# 5. Wartości własne AA_dag (zamieniamy na macierz 1D/wierszową dla tabulate)
vals_AA, _ = eigendecomp(AA_dag, is_hermitian=True)
wypisz_macierz("Wartości własne AA_dag", [vals_AA])

## Zadanie 2: Konstrukcja projektorów
**Opis:** Analiza właściwości projektora dla danego stanu $\ket{\psi}$.

**Dane:** 
$$\ket{\psi} = \frac{1}{\sqrt{3}}\begin{bmatrix} 1 \\ i \\ 1 \end{bmatrix}$$

**Analiza analityczna:**
1. Wyznacz macierz rzutową $P_{\psi}=\ket{\psi}\bra{\psi}$.
2. Udowodnij, że $P_{\psi}$ jest hermitowski ($P_{\psi}=P_{\psi}^{\dagger}$).
3. Udowodnij, że $P_{\psi}$ jest idempotentny ($P_{\psi}^{2}=P_{\psi}$).
4. Oblicz ślad $\operatorname{Tr} P_{\psi}$ (powinien wynosić 1).
5. Zdefiniuj projektor komplementarny $P_{\psi}^{\perp}=I-P_{\psi}$.

#### ✍️ Rozwiązanie analityczne:

1. **Projektor $P_\psi$ (iloczyn zewnętrzny):**
  
2. **Hermitowskość ($P_\psi = P_\psi^\dagger$):**
   
3. **Idempotentność ($P_\psi^2 = P_\psi$):**

4. **Ślad ($\operatorname{Tr} P_\psi$):**
   
5. **Projektor komplementarny:** 

In [ ]:
# Definicja stanu i jego normalizacja
psi_raw = np.array([1, 1j, 1])
# psi wektor znormalizowany
psi = ...

# 1. Wyznaczenie projektora (iloczyn zewnętrzny, uwzględniamy sprzężenie psi)
P_psi = ...
wypisz_macierz("Projektor P_psi", P_psi)

# 2. Sprawdzenie hermitowskośći
is_herm = ...
print(f"Czy P_psi jest hermitowski? {is_herm}")

# 3. Sprawdzenie idempotentności
is_idem = ...
print(f"Czy P_psi jest idempotentny? {is_idem}")

# 4. Obliczenie śladu
print(f"Ślad P_psi: {np.trace(P_psi):.2f}")

# 5. Wyznaczenie projektora komplementarnego
I = ...
P_psi_perp = ...
wypisz_macierz("Projektor komplementarny P_psi_perp", P_psi_perp)

# Dodatkowa weryfikacja właściwości projektora komplementarnego
print(f"Czy P_perp jest hermitowski? {np.allclose(P_psi_perp, P_psi_perp.conj().T)}")
print(f"Czy P_psi * P_perp = 0? {np.allclose(P_psi @ P_psi_perp, 0)}")

## Zadanie 3: Analiza spektralna macierzy $2 \times 2$
**Opis:** Rozkład spektralny macierzy $M$.

**Dane:** 
$$M = \begin{bmatrix} 4 & 2 \\ 2 & 1 \end{bmatrix}$$

**Analiza analityczna:**
1. Wyznacz wartości własne $\lambda_{1}, \lambda_{2}$.
2. Wyznacz i znormalizuj wektory własne $\ket{v_{1}}, \ket{v_{2}}$.
3. Sprawdź ortogonalność: $\langle v_{1}|v_{2}\rangle=0$.
4. Zbuduj projektory $P_{i}=\ket{v_{i}}\bra{v_{i}}$ i wykonaj rozkład spektralny $M = \lambda_1 P_1 + \lambda_2 P_2$.

#### ✍️ Rozwiązanie analityczne:

1. **Wartości własne ($\lambda_1, \lambda_2$):**
   
2. **Wektory własne ($\ket{v_1}, \ket{v_2}$):**
   * **Dla $\lambda_1 = 5$:** 
   * **Dla $\lambda_2 = 0$:** 

3. **Ortogonalność ($\braket{v_1 | v_2} = 0$):**
   
4. **Projektory i rozkład spektralny ($M = \lambda_1 P_1 + \lambda_2 P_2$):**
   

In [ ]:
M = np.array([[4, 2], [2, 1]])
wypisz_macierz("Macierz wyjściowa M", M)

# 1. Obliczenie wartości i wektorów własnych (eigh jest zoptymalizowana dla macierzy symetrycznych)
# Uwaga: eigh sortuje wartości własne rosnąco, czyli: [0., 5.]
eigenvalues, eigenvectors_matrix = ...
print("\nWartości własne M:", eigenvalues)

# 2. Wyodrębnienie wektorów własnych (kolumny macierzy eigenvectors_matrix)
v1 = ...
v2 =...
print("Wektor własny 1 (dla \u03bb=0):", v1)
print("Wektor własny 2 (dla \u03bb=5):", v2)

# 3. Sprawdzenie ortogonalności za pomocą iloczynu skalarnego (np.vdot)
ortho = ...
print(f"Czy wektory są ortogonalne? {ortho}")

# 4. Konstrukcja projektorów przy użyciu iloczynu zewnętrznego (np.outer)
P1 = ...
P2 = ...

wypisz_macierz("Projektor P1 (\u03bb=0)", P1)
wypisz_macierz("Projektor P2 (\u03bb=5)", P2)

# Rekonstrukcja spektralna macierzy M
reconstruction = ...
wypisz_macierz("Zrekonstruowana macierz M", reconstruction)

# Weryfikacja poprawności rozkładu widmowego
print(f"Czy rekonstrukcja zgadza się z oryginalną macierzą M? {np.allclose(reconstruction, M)}")

## Zadanie 4: Relacja zupełności w praktyce
**Opis:** Weryfikacja własności sumy projektorów i własności normy.

**Analiza analityczna:**
1. Użyj projektorów $P_{1}, P_{2}$ z Zadania 3.
2. Oblicz $P_{1}+P_{2}$ i sprawdź, czy jest to macierz tożsamości $I$.
3. Udowodnij ortogonalność projektorów: $P_{1}P_{2}=P_{2}P_{1}=0$.
4. Dla dowolnego wektora $\ket{x} = \begin{bmatrix} a \\ b \end{bmatrix}$, udowodnij, że $\|\ket{x}\|^2 = \|P_{1}\ket{x}\|^2 + \|P_{2}\ket{x}\|^2$.
5. **Pytanie:** Dlaczego $\sum P_{i} = I$ jest kluczowe w pomiarach kwantowych?

#### ✍️ Rozwiązanie analityczne:

1. **Definicja wyjściowych projektorów:**

2. **Weryfikacja relacji zupełności ($\sum P_i = I$):**
   
3. **Ortogonalność projektorów ($P_1 P_2 = P_2 P_1 = 0$):**

4. **Zachowanie normy (Twierdzenie Pitagorasa w przestrzeni Hilberta):**

5. **Znaczenie fizyczne relacji zupełności w pomiarach kwantowych:**


In [ ]:
# 1. Konstrukcja projektorów na podstawie wektorów własnych z poprzedniego zadania
P1 = ...
P2 = ...

# 2. Suma projektorów i weryfikacja relacji zupełności
sum_p = ...
wypisz_macierz("Suma projektorów P1 + P2", sum_p)
print(f"Czy suma to macierz tożsamości I? {np.allclose(sum_p, np.eye(2))}")

# 3. Ortogonalność operatorów rzutowych
p1_p2_zero = ...
p2_p1_zero = ...
print(f"Czy P1 * P2 = 0 oraz P2 * P1 = 0? {p1_p2_zero and p2_p1_zero}")

# 4. Relacja zachowania normy (Twierdzenie Pitagorasa)
x = np.array([1.0, 2.0])
# x_norm_sq -> ||x||^2
x_norm_sq = ...

# Obliczenie kwadratów norm poszczególnych rzutów wektora x
proj1_norm_sq = ...
proj2_norm_sq = ...
suma_rzutow_sq =...

print(f"\nKwadrat normy wektora wejściowego ||x||^2: {x_norm_sq:.2f}")
print(f"Suma kwadratów norm rzutów ||P1 x||^2 + ||P2 x||^2: {suma_rzutow_sq:.2f}")
print(f"Czy relacja zachowania normy jest spełniona? {np.isclose(x_norm_sq, suma_rzutow_sq)}")

## Zadanie 5: Operator unitarny i zachowanie normy
**Opis:** Analiza transformacji unitarnej i reguły Borna.

**Dane:** 
$$U = \frac{1}{\sqrt{2}} \begin{bmatrix} 1 & 1 \\ 1 & -1 \end{bmatrix}$$

**Analiza analityczna:**
1. Wykaż, że $U$ jest macierzą unitarną ($U^{\dagger}U = I$).
2. Oblicz wartości własne i wektory własne $U$.
3. Wykaż, że $U = \lambda_{1}P_{1} + \lambda_{2}P_{2}$.
4. Oblicz stan po transformacji $\ket{\phi} = U\ket{\psi}$ dla $\ket{\psi} = \ket{0} = \begin{bmatrix} 1 \\ 0 \end{bmatrix}$.
5. Oblicz normę $\|\ket{\phi}\|$ i porównaj z $\|\ket{\psi}\|$.
6. Oblicz prawdopodzenia $p_{0}$ i $p_{1}$ przy pomiarze w bazie obliczeniowej (reguła Borna).

#### ✍️ Rozwiązanie analityczne:

1. **Unitarność ($U^\dagger U = I$):**
  
2. **Wartości własne i wektory własne $U$:**
   

3. **Rozkład spektralny ($U = \lambda_1 P_1 + \lambda_2 P_2$):**
   
4. **Transformacja stanu ($\ket{\phi} = U\ket{\psi}$):**
   
5. **Weryfikacja zachowania normy:**
   
6. **Reguła Borna i prawdopodobieństwa pomiaru:**
   

In [ ]:
# Definicja macierzy unitarnej U (Bramka Hadamarda)
U = (1 / np.sqrt(2)) * np.array([[1, 1], [1, -1]])
wypisz_macierz("Macierz unitarna U", U)

# 1. Weryfikacja unitarności (U_dagger @ U = I)
U_dag_U = ...
wypisz_macierz("Iloczyn U_dagger @ U", U_dag_U)
print(f"Czy U jest unitarny? {np.allclose(U_dag_U, np.eye(2))}")

# 2. Wartości własne i wektory własne
# U jest symetryczna/hermitowska, więc eigh gwarantuje stabilność numeryczną
evals, evecs = ...
print("\nWartości własne U:", evals)
print("Wektor własny 1 (kolumna 0):", evecs[:, 0])
print("Wektor własny 2 (kolumna 1):", evecs[:, 1])

# 3. Rekonstrukcja spektralna (Rozkład widmowy)
P_plus = ...
P_minus = ...

wypisz_macierz("Projektor P_plus (dla \u03bb = -1)", P_plus)
wypisz_macierz("Projektor P_minus (dla \u03bb = 1)", P_minus)

# Rekonstrukcja: suma iloczynów wartości własnych i projektorów
reconstruction = ...
wypisz_macierz("Rekonstrukcja spektralna U", reconstruction)
print(f"Czy rekonstrukcja spektralna zgadza się z U? {np.allclose(reconstruction, U)}")

# 4. Transformacja stanu i weryfikacja zachowania normy
psi = np.array([1, 0])
phi = ...

print(f"\nStan po transformacji |phi>: {phi}")
print(f"Norma stanu przed transformacją: {np.linalg.norm(psi):.2f}")
print(f"Norma stanu po transformacji: {np.linalg.norm(phi):.2f}")

# 5. Reguła Borna i rozkład prawdopodobieństwa
probs = ...
print(f"\nPrawdopodobieństwa rzutu na bazę obliczeniową [p0, p1]: {probs}")
print(f"Sumaryczne prawdopodobieństwo: {np.sum(probs):.2f}")